# Caso 2 — Transamerica Airlines: ¿Comprar el Piper Chieftain?

**Tarea 8 — Minería de Datos | Lead University**

---

## Contexto del problema

José Cuervo, presidente de **Transamerica Airlines**, evalúa comprar un **Piper Chieftain** por
$600.000 para ampliar su flota. La aeronave operaría durante **5 años** con una mezcla de
vuelos programados y vuelos charter.

### Parámetros base (estimación central)

| Parámetro | Valor base |
|-----------|------------|
| Precio de compra | $600.000 (pagado al final del año 1) |
| Vida útil del proyecto | 5 años |
| Horas de vuelo al año | 1.000 (esperado), 800 (realista), 700 (pesimista) |
| % vuelos programados | 60% (base), 40%–70% (rango) |
| Ocupación vuelos programados | 60% (base), 50%–70% (rango) |
| Pasajeros máx. por vuelo | 10 |
| Tarifa charter | $1.900/h (base), $1.600–$2.200/h (rango) |
| Tarifa programado | $240/persona/h (base), $200–$300 (rango) |
| Costo operativo | $1.200/h (base), $1.150–$1.250 (rango ±$50) |
| Costos fijos anuales | $160.000 |
| Depreciación anual (lineal 5 años) | $120.000 |
| Tasa impositiva | 33% |
| Tasa de descuento (after-tax) | 15% |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '..')

from scripts import MonteCarloSimulator

np.random.seed(42)

---

## 1. Modelo matemático

### Estructura de ingresos

$$\text{Horas charter} = \text{Horas totales} \times (1 - \%\text{prog})$$

$$\text{Horas programadas} = \text{Horas totales} \times \%\text{prog}$$

$$\text{Ingresos charter} = \text{Horas charter} \times \text{Tarifa charter}$$

$$\text{Ingresos programados} = \text{Horas prog} \times \text{Pasajeros máx} \times \text{Ocupación} \times \text{Tarifa prog}$$

$$\text{Ingresos totales} = \text{Ingresos charter} + \text{Ingresos programados}$$

### Estructura de costos y flujo de caja

$$\text{Costos variables} = \text{Horas totales} \times \text{Costo/hora}$$

$$\text{EBITDA} = \text{Ingresos} - \text{Costos variables} - \text{Costos fijos}$$

$$\text{EBIT} = \text{EBITDA} - \text{Depreciación}$$

$$\text{Impuestos} = \max(0, \text{EBIT}) \times 0.33$$

$$\text{Flujo de caja} = \text{EBITDA} - \text{Impuestos}$$

*(La depreciación se suma de vuelta porque no es una salida de caja real)*

### VAN del proyecto

El pago de $600.000 ocurre al **final del año 1**, por lo que se descuenta:

$$\text{VAN} = -\frac{600.000}{(1+0.15)^1} + \sum_{t=1}^{5} \frac{\text{FC}_t}{(1+0.15)^t}$$

In [ ]:
def calcular_flujo_anual(horas, pct_prog, ocupacion, tarifa_charter,
                         tarifa_prog, costo_hora, costos_fijos=160_000,
                         depreciacion=120_000, tasa_imp=0.33, pasajeros_max=10):
    """Calcula el flujo de caja after-tax de un año de operación."""
    horas_charter  = horas * (1 - pct_prog)
    horas_prog     = horas * pct_prog

    ingresos_charter = horas_charter * tarifa_charter
    ingresos_prog    = horas_prog * pasajeros_max * ocupacion * tarifa_prog
    ingresos_total   = ingresos_charter + ingresos_prog

    costos_var = horas * costo_hora
    ebitda     = ingresos_total - costos_var - costos_fijos
    ebit       = ebitda - depreciacion
    impuestos  = max(0, ebit) * tasa_imp

    flujo_caja = ebitda - impuestos
    return flujo_caja, ingresos_total, ebitda, ebit


def calcular_van(flujos_anuales, precio_compra=600_000, tasa=0.15):
    """VAN con el pago al final del año 1."""
    van = -precio_compra / (1 + tasa)**1
    for t, fc in enumerate(flujos_anuales, start=1):
        van += fc / (1 + tasa)**t
    return van

---

## 2. Modelo determinístico — valores base

In [ ]:
# Valores base (estimación central de José)
horas_base        = 1_000
pct_prog_base     = 0.60
ocupacion_base    = 0.60
tarifa_charter_b  = 1_900
tarifa_prog_b     = 240
costo_hora_b      = 1_200

fc_base, ing_base, ebitda_base, ebit_base = calcular_flujo_anual(
    horas_base, pct_prog_base, ocupacion_base,
    tarifa_charter_b, tarifa_prog_b, costo_hora_b
)

print("=== Año tipo — valores base ===")
print(f"  Ingresos totales:  ${ing_base:>12,.0f}")
print(f"  EBITDA:            ${ebitda_base:>12,.0f}")
print(f"  EBIT:              ${ebit_base:>12,.0f}")
print(f"  Flujo de caja:     ${fc_base:>12,.0f}")

flujos_det = [fc_base] * 5
van_det = calcular_van(flujos_det)

print(f"\nVAN determinístico (valores base): ${van_det:,.0f}")
print("→", "Proyecto VIABLE" if van_det > 0 else "Proyecto NO VIABLE")

### Interpretación — Modelo determinístico

Con los valores base de José (1.000 horas, 60% programados, 60% ocupación, tarifa charter
$1.900/h, tarifa programado $240/persona/h, costo $1.200/h), el modelo puntual indica si
el proyecto crea o destruye valor.

Sin embargo, todos estos parámetros son estimaciones sujetas a incertidumbre. El siguiente
paso es cuantificar el impacto de esa incertidumbre con Monte Carlo.

---

## 3. Variables inciertas y sus distribuciones

| Variable | Distribución | Low | Moda/Central | High |
|----------|-------------|-----|---------|------|
| Horas de vuelo/año | Triangular | 700 | 800 | 1.000 |
| % vuelos programados | Triangular | 0.40 | 0.60 | 0.70 |
| Ocupación vuelos prog. | Triangular | 0.50 | 0.60 | 0.70 |
| Tarifa charter ($/h) | Triangular | 1.600 | 1.900 | 2.200 |
| Tarifa programado ($/persona/h) | Triangular | 200 | 240 | 300 |
| Costo operativo ($/h) | Triangular | 1.150 | 1.200 | 1.250 |

Se usa **distribución triangular** para todas las variables, consistente con el método
del notebook de la Clase 11.

In [ ]:
# Visualizar distribuciones triangulares de las variables inciertas
sim_viz = MonteCarloSimulator(n=1, seed=0)  # solo para gráficos

variables_inciertas = {
    "Horas de vuelo/año":       {"low": 700,   "mode": 800,   "high": 1_000},
    "% vuelos programados":     {"low": 0.40,  "mode": 0.60,  "high": 0.70},
    "Ocupación vuelos prog.": {"low": 0.50,  "mode": 0.60,  "high": 0.70},
    "Tarifa charter ($/h)":    {"low": 1_600, "mode": 1_900, "high": 2_200},
    "Tarifa prog. ($/pers/h)": {"low": 200,   "mode": 240,   "high": 300},
    "Costo operativo ($/h)":   {"low": 1_150, "mode": 1_200, "high": 1_250},
}

for nombre, p in variables_inciertas.items():
    sim_viz.plot_distribucion_triangular(nombre, p["low"], p["mode"], p["high"])
    print(f"{nombre} → low: {p['low']} | mode: {p['mode']} | high: {p['high']}")

---

## 4. Simulación de Monte Carlo — 10.000 iteraciones

In [ ]:
n_sim = 10_000
años  = 5
np.random.seed(42)

# Muestrear cada variable independientemente para cada año y simulación
# Forma: (n_sim, años)
horas_s       = np.random.triangular(700,   800,   1_000, size=(n_sim, años))
pct_prog_s    = np.random.triangular(0.40,  0.60,  0.70,  size=(n_sim, años))
ocupacion_s   = np.random.triangular(0.50,  0.60,  0.70,  size=(n_sim, años))
t_charter_s   = np.random.triangular(1_600, 1_900, 2_200, size=(n_sim, años))
t_prog_s      = np.random.triangular(200,   240,   300,   size=(n_sim, años))
costo_hora_s  = np.random.triangular(1_150, 1_200, 1_250, size=(n_sim, años))

# Cálculo vectorizado
pasajeros_max = 10
costos_fijos  = 160_000
depreciacion  = 120_000
tasa_imp      = 0.33

horas_charter_s  = horas_s * (1 - pct_prog_s)
horas_prog_s     = horas_s * pct_prog_s

ing_charter_s    = horas_charter_s * t_charter_s
ing_prog_s       = horas_prog_s * pasajeros_max * ocupacion_s * t_prog_s
ingresos_s       = ing_charter_s + ing_prog_s

costos_var_s     = horas_s * costo_hora_s
ebitda_s         = ingresos_s - costos_var_s - costos_fijos
ebit_s           = ebitda_s - depreciacion
impuestos_s      = np.where(ebit_s > 0, ebit_s * tasa_imp, 0)
flujos_s         = ebitda_s - impuestos_s    # (n_sim, años)

# VAN: el pago de $600k ocurre al final del año 1
factores = np.array([1 / (1.15)**t for t in range(1, años + 1)])  # (años,)
van_s = -600_000 / 1.15 + (flujos_s * factores).sum(axis=1)      # (n_sim,)

print(f"Simulaciones: {n_sim}")
print(f"Primeros 5 VAN: {van_s[:5]}")

In [ ]:
# Registrar en el simulador para análisis
sim = MonteCarloSimulator(n=n_sim, seed=42)

# Variables inciertas (promedios a 5 años para el análisis de correlación)
sim.muestras["horas_vuelo"]      = horas_s.mean(axis=1)
sim.muestras["pct_programados"]  = pct_prog_s.mean(axis=1)
sim.muestras["ocupacion"]        = ocupacion_s.mean(axis=1)
sim.muestras["tarifa_charter"]   = t_charter_s.mean(axis=1)
sim.muestras["tarifa_prog"]      = t_prog_s.mean(axis=1)
sim.muestras["costo_hora"]       = costo_hora_s.mean(axis=1)

sim.agregar_resultado("VAN", van_s)

sim.resumen("VAN")

### Interpretación — Estadísticas del VAN simulado

La media, mediana y percentiles del VAN resumen el resultado esperado del proyecto bajo
incertidumbre. La probabilidad de pérdida (VAN < 0) es la métrica clave de riesgo:
indica en qué fracción de los escenarios el proyecto destruiría valor para Transamerica.

---

## 5. Análisis de resultados

In [ ]:
# Histograma del VAN
sim.plot_histograma("VAN", xlabel="VAN ($)",
                   titulo="Distribución del VAN — Transamerica Airlines (Monte Carlo)")

In [ ]:
# Probabilidad acumulada del VAN
sim.plot_probabilidad_acumulada("VAN", xlabel="VAN ($)")

In [ ]:
# Correlación de cada variable con el VAN
sim.plot_correlacion("VAN")

### Interpretación — Correlaciones con el VAN

El gráfico de correlaciones identifica cuáles son los **factores que más impactan el VAN**.
Las variables con correlación alta (cercana a 1 o −1) son los riesgos críticos del proyecto:

- Variables de **ingreso** (tarifa charter, tarifa programado, horas de vuelo, ocupación)
  tendrán correlación positiva: a mayor valor, mayor VAN.
- El **costo operativo** tendrá correlación negativa: a mayor costo, menor VAN.
- El **% de vuelos programados** puede tener correlación positiva o negativa dependiendo
  de qué modalidad es más rentable en las condiciones del mercado.

In [ ]:
# Impacto por rangos (análisis de sensibilidad)
sim.plot_impacto_por_rangos("VAN")

### Interpretación — Impacto por rangos

El análisis por rangos (bajo / medio / alto) complementa la correlación: muestra el
**impacto absoluto en dólares** al pasar de un nivel bajo a uno alto en cada variable.
Esto facilita priorizar qué variables merece gestionar activamente:

- Si el impacto de pasar de tarifa charter baja a alta supera los $200k en VAN, negociar
  precios charter es prioritario.
- Si el impacto de las horas de vuelo es también alto, maximizar la utilización del avión
  es igualmente crítico.

---

## 6. Probabilidades clave de decisión

In [ ]:
p_positivo = (van_s >= 0).mean()
p_negativo = (van_s < 0).mean()

print(f"P(VAN ≥ 0) = {p_positivo:.1%}  → probabilidad de que el proyecto sea rentable")
print(f"P(VAN < 0) = {p_negativo:.1%}  → probabilidad de pérdida")
print()
print(f"VAN mínimo:    ${van_s.min():>12,.0f}")
print(f"VAN máximo:    ${van_s.max():>12,.0f}")
print(f"VAN promedio:  ${van_s.mean():>12,.0f}")
print(f"VAN mediana:   ${np.median(van_s):>12,.0f}")
print()
print(f"Percentil  5%: ${np.percentile(van_s, 5):>12,.0f}")
print(f"Percentil 25%: ${np.percentile(van_s, 25):>12,.0f}")
print(f"Percentil 75%: ${np.percentile(van_s, 75):>12,.0f}")
print(f"Percentil 95%: ${np.percentile(van_s, 95):>12,.0f}")

In [ ]:
# Análisis de escenarios extremos (determinístico)
print("=== Análisis de escenarios extremos ===")

escenarios = {
    "Optimista (1000h, 70% prog, 70% ocup, $2200 charter, $300 prog, $1150 costo)": {
        "horas": 1_000, "pct": 0.70, "ocup": 0.70,
        "tc": 2_200, "tp": 300, "co": 1_150
    },
    "Base (1000h, 60% prog, 60% ocup, $1900 charter, $240 prog, $1200 costo)": {
        "horas": 1_000, "pct": 0.60, "ocup": 0.60,
        "tc": 1_900, "tp": 240, "co": 1_200
    },
    "Realista (800h, 60% prog, 60% ocup, $1900 charter, $240 prog, $1200 costo)": {
        "horas": 800,  "pct": 0.60, "ocup": 0.60,
        "tc": 1_900, "tp": 240, "co": 1_200
    },
    "Pesimista (700h, 40% prog, 50% ocup, $1600 charter, $200 prog, $1250 costo)": {
        "horas": 700,  "pct": 0.40, "ocup": 0.50,
        "tc": 1_600, "tp": 200, "co": 1_250
    },
}

for nombre, p in escenarios.items():
    fc, _, _, _ = calcular_flujo_anual(
        p["horas"], p["pct"], p["ocup"], p["tc"], p["tp"], p["co"]
    )
    van = calcular_van([fc] * 5)
    print(f"\n{nombre}")
    print(f"  Flujo anual: ${fc:,.0f}  |  VAN: ${van:,.0f}  → {'VIABLE' if van > 0 else 'NO VIABLE'}")

---

## 7. Respuestas a las preguntas del caso

### Pregunta 1 — ¿Debería José Cuervo comprar el Piper Chieftain?

La respuesta depende del perfil de riesgo del inversionista y de los resultados de la
simulación:

- Si el **VAN esperado es positivo** (media > 0) y la **probabilidad de pérdida es baja**
  (P(VAN < 0) < 20%), la recomendación es **comprar**: el proyecto crea valor en la mayoría
  de los escenarios.
- Si el VAN esperado es positivo pero la probabilidad de pérdida es moderada o alta, José
  debería negociar mejores condiciones (precio de compra, tarifas mínimas garantizadas) antes
  de comprometerse.
- Si incluso el escenario optimista produce un VAN negativo, la compra no se justifica.

**Factores a favor de la compra:**
- José ya tiene experiencia operativa y relaciones en el mercado.
- La mezcla charter/programado ha resultado rentable con su flota existente.
- El avión ya tiene todos los equipos requeridos (sin inversión adicional en aviónica).

**Factores de riesgo a monitorear:**
- Las tarifas charter son muy sensibles a la competencia ($1.600–$2.200/h = rango del 37%).
- Las horas de vuelo dependen del estado de la economía (700–1.000 = rango del 43%).
- El pago de $600.000 al final del año 1 compromete liquidez antes de generar retornos.

### Pregunta 2 — Factores con mayor impacto en la rentabilidad

Según el análisis de correlaciones y rangos, los factores críticos son (en orden típico):

1. **Tarifa charter** — mayor variabilidad en dólares absolutos; controlada por el mercado.
2. **Horas de vuelo** — factor multiplicador de todos los ingresos; depende de la economía.
3. **Ocupación en vuelos programados** — afecta directamente los ingresos por pasajero.
4. **Tarifa programado** — menor volatilidad que el charter pero impacto significativo.
5. **Costo operativo** — variación pequeña (±$50/h) pero sobre 800–1.000 horas acumula.
6. **% vuelos programados** — su impacto neto depende del diferencial de rentabilidad
   entre charter y programado.

La **simulación de Monte Carlo** permite a José ver simultáneamente el efecto combinado
de todas estas incertidumbres, algo imposible con el análisis determinístico.